#                                                               WEEK-07 DELTA LAKE ASSIGNMENT 

## Objective

The objective of this assignment is to demonstrate an end-to-end Delta Lake workflow using Apache Spark. The process includes loading a CSV dataset, performing basic data cleaning, storing the cleaned data as a Delta table, creating an incremental dataset, applying merge operations, and validating the final output.

This assignment demonstrates how Delta Lake efficiently handles incremental data while maintaining data consistency.


## STEP 1 - Load the Dataset

The Superstore dataset is loaded into Apache Spark from Unity Catalog Volume. The schema and sample records are verified before any transformation is performed.

In [0]:
from pyspark.sql import functions as F

# Load raw Superstore data from Unity Catalog Volume
df_raw = spark.read.csv(
    "/Volumes/databricksmaster/default/superstore_data/Sample - Superstore.csv",
    header=True,
    inferSchema=True,
)
# Display the total number of records.
print("Raw rows:", df_raw.count())
df_raw.printSchema()
df_raw.show(5)

Raw rows: 9994
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+------

## STEP 2 - Data Cleaning

The dataset is examined for null values and duplicate records. This step ensures that the dataset is clean before loading it into a Delta table.

In [0]:
from pyspark.sql import functions as F


# Count null values in each column
null_counts = df_raw.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_raw.columns]
)

print("Null values in each column:")
null_counts.show()
# Remove duplicate rows
df_clean = df_raw.dropDuplicates()
original_rows = df_raw.count()
clean_rows = df_clean.count()

print("Original rows:", original_rows)
print("Rows after removing duplicates:", clean_rows)
print("Clean rows:", clean_rows)
df_clean.show(5)/Volumes/databricksmaster/default/superstore_data/Sample - Superstore.csv

Null values in each column:
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------

### Observation

- No null values were found in the dataset.
- No duplicate records were identified.
- The dataset was already clean and ready for further processing.

## Step 3 - Load Data into a Delta Table

The cleaned dataset is stored in Delta format. This provides ACID transactions, schema enforcement, and supports efficient update and merge operations.

In [0]:
# Save the cleaned dataset as a Delta table
df_clean.write.format("delta").mode("overwrite").save(
    "/Volumes/databricksmaster/default/superstore_data/customer_master"
)

In [0]:
# Read the Delta table
df_master = spark.read.format("delta").load(
    "/Volumes/databricksmaster/default/superstore_data/customer_master"
    )
print("Rows in Delta Table:", df_master.count())

df_master.printSchema()

df_master.show(5)

Rows in Delta Table: 9994
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- Discount: string (nullable = true)
 |-- Profit: double (nullable = true)

+------+--------------+----------+----------+--------------+-----------+------------------+--------+-------------+---------

## Step 4 - Create Incremental Dataset

A sample incremental dataset is created to simulate incoming records. It contains both updated existing records and new records that will be merged into the Delta table.

In [0]:
from pyspark.sql import functions as F

updated_records = df_master.limit(5).withColumn(
    "Sales",
    F.expr("try_cast(Sales as double) + 100D"),
).withColumn(
    "Profit",
    F.expr("try_cast(Profit as double) + 20D"),
)
# Create new records by assigning new Row IDs.
new_records = df_master.limit(5)
new_records = new_records.withColumn(
    "Row ID",
    F.col("Row ID") + 10000,
).withColumn(
    "Sales",
    F.expr("try_cast(Sales as double)"),
)
incremental_df = updated_records.union(new_records)
print("Incremental rows:", incremental_df.count())

incremental_df.show()



Incremental rows: 10
+------+--------------+----------+----------+--------------+-----------+------------------+--------+-------------+---------------+----------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name| Segment|      Country|           City|     State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|  Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+--------+-------------+---------------+----------+-----------+-------+---------------+---------------+------------+--------------------+-------+--------+--------+--------+
|     5|US-2015-108966|2015-10-11|2015-10-18|Standard Class|   SO-20335|    Sean O'Donnell|Consumer|United States|Fort Lauderdale|   Florida|      33311|  South|OFF-ST-10000760|Office Suppli

## STEP 5 - Merge Incremental Data

The Delta Lake MERGE operation updates matching records and inserts new records into the existing Delta table. This demonstrates an incremental data loading process.

In [0]:
from delta.tables import DeltaTable

# Load the Delta table
delta_table = DeltaTable.forPath(
    spark, "/Volumes/databricksmaster/default/superstore_data/customer_master"
)
# Merge incremental data into the Delta table
(
    delta_table.alias("target")
    .merge(incremental_df.alias("source"), "target.`Row ID` = source.`Row ID`")
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

merged_df = spark.read.format("delta").load(
    "/Volumes/databricksmaster/default/superstore_data/customer_master"
)

print("Rows after merge:", merged_df.count())
merged_df.show(10)

Rows after merge: 9999
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|    Customer Name|    Segment|      Country|         City|         State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+--------------+-----------+-------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|   140|CA-2016-145583|2016-10-13|2016-10-19|Standard Class|   LC-16885|   Lena Creighton|   Consumer|United States|    Roseville|    California|      95661|   West|FUR-FU-1

## STEP 6 - Validation

The final dataset is validated to ensure that the merge operation completed successfully. Record count, duplicate records, and inserted records are verified.

In [0]:
# Check the total number of records after the merge
print("Total rows after merge:", merged_df.count())
from pyspark.sql import functions as F

duplicate_rows = merged_df.groupBy("Row ID").count().filter(F.col("count") > 1)

print("Duplicate Row IDs:")
duplicate_rows.show()
merged_df.filter(F.col("Row ID") > 10000).show()
merged_df.orderBy("Row ID").select("Row ID", "Sales", "Profit").show(10, truncate=False)

Total rows after merge: 9999
Duplicate Row IDs:
+------+-----+
|Row ID|count|
+------+-----+
+------+-----+

+------+--------------+----------+----------+--------------+-----------+------------------+--------+-------------+---------------+----------+-----------+-------+---------------+---------------+------------+--------------------+------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|     Customer Name| Segment|      Country|           City|     State|Postal Code| Region|     Product ID|       Category|Sub-Category|        Product Name| Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+------------------+--------+-------------+---------------+----------+-----------+-------+---------------+---------------+------------+--------------------+------+--------+--------+--------+
| 10005|US-2015-108966|2015-10-11|2015-10-18|Standard Class|   SO-20335|    Sean O'Donnell|Consumer|Unite

In [0]:
# Export the master dataset as a CSV file
spark.read.format("delta").load(
    "/Volumes/databricksmaster/default/superstore_data/customer_master"
).toPandas().to_csv(
    "/Volumes/databricksmaster/default/superstore_data/customer_master.csv", index=False
)

print("customer_master.csv created successfully.")

customer_master.csv created successfully.


In [0]:
# Export the incremental dataset as a CSV file
incremental_df.toPandas().to_csv(
    "/Volumes/databricksmaster/default/superstore_data/customer_incremental.csv",
        index=False
        )

print("customer_incremental.csv created successfully.")

customer_incremental.csv created successfully.


# STEP 7 - Conclusion

### Summary

This assignment demonstrated a complete Delta Lake workflow using Apache Spark.

The Superstore dataset was successfully loaded and examined for data quality. The dataset was found to be free from null values and duplicate records and was stored as a Delta table.

An incremental dataset was then created to simulate real-world data ingestion. Using Delta Lake's MERGE operation, existing records were updated while new records were inserted into the Delta table.

Finally, validation checks confirmed that the merge operation completed successfully, the record count was as expected, and no duplicate Row IDs were present.

This workflow demonstrates how Delta Lake can be used to efficiently manage incremental data loading while maintaining data consistency and reliability.

In [0]:
print("Final Record Count :", merged_df.count())
print("Total Columns :", len(merged_df.columns))

print("Assignment completed successfully.")

Final Record Count : 9999
Total Columns : 21
Assignment completed successfully.
